In [1]:
from kerops.settings.autotune import autotune
import torch

In [2]:
def generate_inputs(problem_sizes):
    in_channels, out_channels = problem_sizes['in_channels'], problem_sizes['out_channels']

    if in_channels <= 32 and out_channels <= 32:
        base = 128
    elif in_channels <= 64 and out_channels <= 64:
        base = 96
    else:
        base = 64

    x = torch.randn(1, in_channels, base, base, base, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
    w = torch.randn(3, 3, 3, in_channels, out_channels, device='cuda', dtype=torch.float16)

    return x, w

problem_size_names = ['in_channels', 'out_channels']

channels = [2 ** i for i in range(4, 8)]
problem_sizes = [{'in_channels': cin, 'out_channels': cout} for cin in channels for cout in channels if (cin == 2 * cout) or (cin * 2 == cout) or (cin == cout)]

def pruning_rule(problem_size, named_config):
    D_BLOCK = named_config['D_BLOCK']
    CIN_BLOCK = named_config['CIN_BLOCK']

    in_channels, out_channels = problem_size['in_channels'], problem_size['out_channels']

    if in_channels >= 32 and out_channels >= 32 and D_BLOCK > 32:
        return False

    if (in_channels >= 128 or out_channels >= 128) and D_BLOCK > 16:
        return False

    if CIN_BLOCK > in_channels:
        return False

    return True

In [4]:
from kerops.ops.conv import Conv3d
from kerops.settings import TableKernelConfig, ConfiguredFunction


autotune(
    Conv3d,
    generate_inputs,
    problem_sizes,
    pruning_rule,
    toml_path='test4.toml',
    n_iters=100,
    D_BLOCK=[16, 32, 64],
    num_warps=[2, 4],
    CIN_BLOCK=[16, 32, 64],
    WEIGHT_MAJOR=[True, False],
    LOAD_WEIGHT_FIRST=[True]
)

Problem sizes:   0%|          | 0/10 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/12 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/12 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/12 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/12 [00:00<?, ?it/s]

...

Benchmark configs:   0%|          | 0/12 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/12 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/12 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/12 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/12 [00:00<?, ?it/s]

In [1]:
from kerops.ops.conv.conv import Conv3d, autotune_conv, autotune_bnreluconv
from kerops.ops.assets import ASSETS_ROOT

autotune_bnreluconv(ASSETS_ROOT / "BNReLUConv3d.toml", n_iters=100)

Problem sizes:   0%|          | 0/10 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/6 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/6 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/6 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/6 [00:00<?, ?it/s]

...

Benchmark configs:   0%|          | 0/6 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/6 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/6 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/6 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/6 [00:00<?, ?it/s]

In [7]:
from kerops.ops.conv import Conv3d
from kerops.settings import TableKernelConfig, ConfiguredFunction

conv3dconfig = TableKernelConfig(
    problem_size_names=problem_size_names,
    confarg_names=['D_BLOCK', 'num_warps', 'CIN_BLOCK', 'WEIGHT_MAJOR', 'LOAD_WEIGHT_FIRST'],
    args_to_problem_sizes=lambda weight: tuple(weight.shape[-2:]),
    toml_path='test.toml'
)

Conv3dconf = ConfiguredFunction(Conv3d, conv3dconfig)

In [6]:
from kerops.ops.conv.conv import Conv3d, generate_inputs_conv

In [45]:
in_channels = 128
out_channels = 64

problem_size = {'in_channels': in_channels, 'out_channels': out_channels}

x, w = generate_inputs_conv(problem_size)

In [46]:
import torch

In [50]:
%%timeit -r 20 -n 20
Conv3d(x, w)
torch.cuda.synchronize()

1.77 ms ± 18.9 μs per loop (mean ± std. dev. of 20 runs, 20 loops each)


In [51]:
%%timeit -r 20 -n 20
Conv3d(x, w, num_warps=2)
torch.cuda.synchronize()

1.76 ms ± 18 μs per loop (mean ± std. dev. of 20 runs, 20 loops each)


In [31]:
set(['a', 'b', 'c', 'asdsds']) - set(['a', 'b', 'c', 'asdsds'])

set()

In [33]:
set([1, 2, 3]).pop()

1